# Project: Fuel Efficiency Prediction
## Phase 1: Data cleaning and EDA (Exploratory Data Analysis)
**Objective:** Clean and prepare the UCI Automobile dataset (1985) to build a predictive model for fuel efficiency (`city-mpg` and `highway-mpg`). 
This analysis focuses on handling missing values and engineering features relevant to automotive design and performance.

* **Dataset Source:** Schlimmer, J. (1985). Automobile [Dataset]. UCI Machine Learning Repository.

In [2]:
#1. Importing necessary libraries

import pandas as pd
import numpy as np

### 1. Data Ingestion
The original dataset lacks headers and uses `?` to represent missing values. We will define the schema based on the `.names` documentation and parse the nulls during ingestion.

In [ ]:


columns = [
    "symboling", "normalized-losses", "make", "fuel-type", "aspiration",
    "num-of-doors", "body-style", "drive-wheels", "engine-location",
    "wheel-base", "length", "width", "height", "curb-weight",
    "engine-type", "num-of-cylinders", "engine-size", "fuel-system",
    "bore", "stroke", "compression-ratio", "horsepower", "peak-rpm",
    "city-mpg", "highway-mpg", "price"
]

# 2 Load the data mapping the column names to the dataset and treating "?" as NaN values
df = pd.read_csv("../data/automobile/imports-85.data", names=columns, na_values="?")

#Display the first few rows of the dataset to verify that it has been loaded correctly
df.head()

,symboling,normalized-losses,make,fuel-type,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,...,engine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price
0,3,NaN,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,13495.0
1,3,NaN,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,16500.0
2,1,NaN,alfa-romero,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154.0,5000.0,19,26,16500.0
3,2,164.0,audi,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102.0,5500.0,24,30,13950.0
4,2,164.0,audi,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115.0,5500.0,18,22,17450.0


### 2. Initial Data Inspection
Before modifying the dataset, we need to understand the data types and the volume of missing information.

In [6]:
# Check data types and non-null counts
print("--- Data Info ---")
df.info()

print("\n--- Missing Values Count ---")
missing_data = df.isnull().sum()
print(missing_data[missing_data > 0]) # Only display columns with missing values

--- Data Info ---
<class 'pandas.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 26 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   symboling          205 non-null    int64  
 1   normalized-losses  164 non-null    float64
 2   make               205 non-null    str    
 3   fuel-type          205 non-null    str    
 4   aspiration         205 non-null    str    
 5   num-of-doors       203 non-null    str    
 6   body-style         205 non-null    str    
 7   drive-wheels       205 non-null    str    
 8   engine-location    205 non-null    str    
 9   wheel-base         205 non-null    float64
 10  length             205 non-null    float64
 11  width              205 non-null    float64
 12  height             205 non-null    float64
 13  curb-weight        205 non-null    int64  
 14  engine-type        205 non-null    str    
 15  num-of-cylinders   205 non-null    str    
 16  engine-size        

### 3. Missing Value Handling Strategy
Based on the domain logic, we will apply different strategies depending on the variable's impact on fuel efficiency:

1. **`normalized-losses`**: This is an insurance risk metric, not a mechanical feature. We will impute missing values using the mean of the respective car `make` to preserve rows without distorting mechanical data. Remaining nulls will be filled with the global mean.
2. **`horsepower`**: A critical engine specification directly tied to fuel consumption. Since only 2 rows are missing, we will drop them to avoid introducing synthetic noise into the predictive model.

In [7]:
#Strategy 1: Impute 'noralized-lossses'
#Fill null values with the mean of the 'normalized-losses' column for each 'make' category
df['normalized-losses'] = df.groupby('make')['normalized-losses'].transform(lambda x: x.fillna(x.mean()))

#Fill any remaining nulls (for makes where all values were null) with the global mean
global_mean_losses = df['normalized-losses'].mean()
df['normalized-losses'] = df['normalized-losses'].fillna(global_mean_losses)

#Strategy 2: Drop critical mechanical rows with missing values in 'horsepower'
df.dropna(subset=['horsepower'], inplace=True)

#Reset index after dropping rows
df.reset_index(drop=True, inplace=True)

#Verifify that there are no more missing values in the critical columns
print("Remaining nulls in horsepower:", df['horsepower'].isnull().sum())
print("Remaining nulls in normalized-losses:", df['normalized-losses'].isnull().sum())


Remaining nulls in horsepower: 0
Remaining nulls in normalized-losses: 0
